# 04 PyTorch Model Evaluation v1

PyTorch evaluation report parity for GreenSpace CNN. This notebook mirrors the TensorFlow NB04 outputs for loss monitoring, threshold tuning, and split-wise reports.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp')

print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN


In [2]:
import numpy as np
import pandas as pd
import torch

from src_torch.config import TORCH_DATA_CONFIG
from src_torch.data import load_split_dfs, resolve_split_schema
from src_torch.evaluation import (
    evaluate_all_splits,
    evaluate_loss_monitoring,
    find_latest_pytorch_checkpoint,
    infer_run_tag_and_variant,
    load_torch_checkpoint_model,
    predict_split,
    save_evaluation_outputs,
    tune_thresholds_f1,
)
from src_torch.training import resolve_device

split_dfs = load_split_dfs()
train_df = split_dfs['train']
val_df = split_dfs['val']
test_df = split_dfs['test']
schema = resolve_split_schema(train_df)
binary_cols = schema.binary_cols
bin_names = schema.bin_names

print('torch:', torch.__version__)
print('Loaded splits:', {k: len(v) for k, v in split_dfs.items()})
print('Binary labels:', binary_cols)

torch: 2.10.0
Loaded splits: {'train': 3145, 'val': 1048, 'test': 1049}
Binary labels: ['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']


## Load PyTorch Checkpoint

In [3]:
USE_MANUAL_MODEL = True
MANUAL_MODEL_PATH = '/Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/best_mcmae_PyTorch_20260614_220926.pt'
## /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260608_220356/best_mcmae_PyTorch_20260608_220356.pt
if USE_MANUAL_MODEL:
    model_path = Path(MANUAL_MODEL_PATH)
else:
    model_path = find_latest_pytorch_checkpoint(preferred_variant='best_mcmae')

assert model_path.exists(), f'Missing PyTorch checkpoint: {model_path}'
run_tag, EVAL_VARIANT = infer_run_tag_and_variant(model_path)
device = resolve_device('auto')
model, model_config, checkpoint = load_torch_checkpoint_model(model_path, device=device)

print('Loaded checkpoint:', model_path)
print('run_tag:', run_tag)
print('EVAL_VARIANT:', EVAL_VARIANT)
print('device:', device)

Loaded checkpoint: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260614_220926/best_mcmae_PyTorch_20260614_220926.pt
run_tag: PyTorch_20260614_220926
EVAL_VARIANT: best_mcmae
device: mps


## Predict Splits

In [4]:
# Set to an integer for quick debug; use None for full split reports.
EVAL_MAX_BATCHES = None
EVAL_BATCH_SIZE = TORCH_DATA_CONFIG['batch_size']

predictions_by_split = {}
for split in ('train', 'val', 'test'):
    print(f'Predicting {split}...')
    predictions_by_split[split] = predict_split(
        model,
        split,
        device=device,
        batch_size=EVAL_BATCH_SIZE,
        max_batches=EVAL_MAX_BATCHES,
    )
    df_used, preds, loss_metrics = predictions_by_split[split]
    print(split, 'n=', len(df_used), 'loss=', round(loss_metrics['loss'], 4))

Predicting train...
train n= 3145 loss= 1.2101
Predicting val...
val n= 1048 loss= 2.0269
Predicting test...
test n= 1049 loss= 2.1305


## Loss Monitoring

In [5]:
loss_monitor_df = evaluate_loss_monitoring(predictions_by_split, binary_cols)
display(loss_monitor_df.round(4))

,split,loss,bin_head_loss,shade_head_loss,score_head_loss,veg_head_loss,bin_head_binary_accuracy,bin_head_pr_auc,shade_head_sparse_categorical_accuracy,score_head_mae,veg_head_mae
0,train,1.2101,0.2061,0.3068,0.4021,0.2951,0.8985,0.9272,0.8529,0.4021,0.2951
1,val,2.0269,0.2534,0.8064,0.5676,0.3995,0.8798,0.8801,0.6746,0.5676,0.3995
2,test,2.1305,0.2646,0.8615,0.5748,0.4296,0.8730,0.8876,0.6578,0.5748,0.4296


## Threshold Tuning

In [6]:
val_used_df, val_preds, _ = predictions_by_split['val']
hard_names_val = [name for name in bin_names if name in val_used_df.columns]
if hard_names_val:
    y_bin_val_true = val_used_df[hard_names_val].fillna(0).astype(int).values
    pred_bin_val_aligned = np.stack([val_preds['bin_head'][:, bin_names.index(name)] for name in hard_names_val], axis=1)
    label_names = hard_names_val
else:
    y_bin_val_true = (val_used_df[binary_cols].fillna(0.0).astype(np.float32).values >= 0.5).astype(int)
    pred_bin_val_aligned = val_preds['bin_head']
    label_names = bin_names

thresholds_df = tune_thresholds_f1(y_bin_val_true, pred_bin_val_aligned, label_names)
best_thresholds = {
    row['label']: float(row['best_threshold'])
    for _, row in thresholds_df.iterrows()
    if np.isfinite(row['best_threshold'])
}

print('Tuned thresholds:', len(best_thresholds), '/', len(label_names))
display(thresholds_df.sort_values('best_f1', ascending=False).reset_index(drop=True).round(4))

Tuned thresholds: 7 / 7


,label,best_threshold,best_f1,best_precision,best_recall,pos_rate,n_pos,n,note
0,multipurpose_open_area,0.4831,0.9460,0.9240,0.9690,0.8015,840,1048,
1,walking_paths,0.3421,0.9301,0.8913,0.9725,0.7643,801,1048,
2,sports_field,0.2942,0.8964,0.8746,0.9194,0.2605,273,1048,
3,built_structures,0.4803,0.8565,0.8527,0.8603,0.4303,451,1048,
4,parking_lots,0.4323,0.8407,0.8195,0.8629,0.3063,321,1048,
5,water_feature,0.4555,0.7890,0.7869,0.7912,0.1737,182,1048,
6,children_s_playground,0.3701,0.6026,0.5697,0.6395,0.1403,147,1048,


## Full Split Evaluation

In [7]:
overall_split_df, per_label_split_df = evaluate_all_splits(
    predictions_by_split,
    binary_cols,
    best_thresholds,
)

print('Overall metrics by split')
display(overall_split_df.round(4))
print('Per-label metrics by split')
display(per_label_split_df.round(4))

Overall metrics by split


,split,n_samples,n_labels,overall_F1@0.5,overall_ROC_AUC,overall_PR_AUC,overall_F1@tuned,shade_acc_overall,shade_acc_conditional,shade_acc_paths_present,score_acc,veg_acc,score_mae,veg_mae,score_ce,veg_ce,score_mae_mean,veg_mae_mean
0,train,3145,7,0.8662,0.9626,0.9272,0.8666,0.8528,0.9270,0.9248,0.6289,0.7803,0.4845,0.3260,NaN,NaN,0.4014,0.2951
1,val,1048,7,0.8242,0.9413,0.8801,0.8373,0.6746,0.6863,0.6841,0.4933,0.6679,0.6316,0.4298,NaN,NaN,0.5676,0.3995
2,test,1049,7,0.8251,0.9392,0.8876,0.8235,0.6597,0.6682,0.6646,0.5091,0.6616,0.6274,0.4514,NaN,NaN,0.5762,0.4298


Per-label metrics by split


,split,label,support_pos,support_neg,P@0.5,R@0.5,F1@0.5,ROC_AUC,PR_AUC,thr_val,P@thr,R@thr,F1@thr
0,train,multipurpose_open_area,2499,646,0.9387,0.9736,0.9558,0.9575,0.9869,0.4831,0.9363,0.9764,0.9559
1,train,walking_paths,2408,737,0.9271,0.9398,0.9334,0.9476,0.9816,0.3421,0.9023,0.9664,0.9332
2,train,sports_field,838,2307,0.9166,0.8914,0.9038,0.9830,0.9647,0.2942,0.8697,0.9320,0.8998
3,train,parking_lots,957,2188,0.9036,0.9007,0.9021,0.9809,0.9594,0.4323,0.8718,0.9164,0.8935
4,train,built_structures,1315,1830,0.8762,0.8829,0.8795,0.9621,0.9498,0.4803,0.8715,0.8875,0.8794
5,train,water_feature,640,2505,0.8412,0.7781,0.8084,0.9514,0.8817,0.4555,0.8255,0.7984,0.8118
6,train,children_s_playground,391,2754,0.6857,0.6752,0.6804,0.9558,0.7666,0.3701,0.6085,0.8031,0.6924
7,val,multipurpose_open_area,840,208,0.9267,0.9631,0.9445,0.9435,0.9824,0.4831,0.9240,0.9690,0.9460
8,val,walking_paths,801,247,0.9109,0.9313,0.9210,0.9323,0.9711,0.3421,0.8913,0.9725,0.9301
9,val,sports_field,273,775,0.9073,0.8608,0.8835,0.9787,0.9405,0.2942,0.8746,0.9194,0.8964


## Save Evaluation Artifacts

In [8]:
saved_paths = save_evaluation_outputs(
    run_tag=run_tag,
    variant=EVAL_VARIANT,
    loss_monitor_df=loss_monitor_df,
    thresholds_df=thresholds_df,
    overall_df=overall_split_df,
    per_label_df=per_label_split_df,
)

for name, path in saved_paths.items():
    print(name, '->', path)

loss_monitor -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/runs/PyTorch_20260614_220926/loss_monitor_best_mcmae.csv
thresholds -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/runs/PyTorch_20260614_220926/thresholds_best_mcmae.csv
overall -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/runs/PyTorch_20260614_220926/overall_metrics_by_split_best_mcmae.csv
per_label -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/runs/PyTorch_20260614_220926/per_label_metrics_by_split_best_mcmae.csv
